# F-OAL Official Code Reproduction on CIFAR-100 (A19 — Checkpoint 3 Compliance)

- **Paper:** *F-OAL: Forward-only Online Analytic Learning with Fast Training and Low Memory Footprint in Class Incremental Learning* (NeurIPS 2024)
- **Author Official Repository:** [`https://github.com/liuyuchen-cz/F-OAL`](https://github.com/liuyuchen-cz/F-OAL) (Pinned Commit: `73e4ac1b4c965030973a2b77f05c57c2f44f49f5`)
- **Group Repository:** [`https://github.com/caoban123/F-OAL`](https://github.com/caoban123/F-OAL)
- **Purpose:** Execute the unmodified official F-OAL implementation on CIFAR-100 (10 tasks x 10 classes, online one-pass stream, batch size 10), extract standardized metrics (`metrics.json`, `metadata.json`, `task_accuracy.csv`), log to Weights & Biases, and package verified raw artifacts into `/content/foal_official_cifar100_results.zip`.


In [ ]:
# === CELL 1: Environment Preflight ===
import os
import sys
import shutil
import platform
import subprocess
import torch

print(f"Python version   : {sys.version}")
print(f"Platform         : {platform.platform()}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "FATAL: CUDA GPU is not available! Please switch Colab/Kaggle runtime to GPU (T4 / P100 / L4 / A100) before running."
    )

device_name = torch.cuda.get_device_name(0)
device_props = torch.cuda.get_device_properties(0)
vram_gb = device_props.total_memory / (1024 ** 3)
print(f"GPU Device       : {device_name}")
print(f"CUDA version     : {torch.version.cuda}")
print(f"Total VRAM       : {vram_gb:.2f} GB")
print(f"Compute Cap.     : {device_props.major}.{device_props.minor}")

total_disk, used_disk, free_disk = shutil.disk_usage("/")
print(f"Free Disk Space  : {free_disk / (1024 ** 3):.2f} GB / {total_disk / (1024 ** 3):.2f} GB")

print("\n--- nvidia-smi ---")
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
# === CELL 2: Clone Official Author Repository (Idempotent & Pinned) ===
import json
import pathlib
import shutil
import subprocess

OFFICIAL_REPO_URL = "https://github.com/liuyuchen-cz/F-OAL.git"
PINNED_COMMIT_SHA = "73e4ac1b4c965030973a2b77f05c57c2f44f49f5"
BASE_DIR = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
OFFICIAL_REPO_DIR = BASE_DIR / "official-F-OAL"
RESULTS_OUT_DIR = BASE_DIR / "official_foal_results"
RESULTS_OUT_DIR.mkdir(parents=True, exist_ok=True)

if not (OFFICIAL_REPO_DIR / ".git").exists():
    if OFFICIAL_REPO_DIR.exists():
        shutil.rmtree(OFFICIAL_REPO_DIR)
    print(f"Cloning {OFFICIAL_REPO_URL} into {OFFICIAL_REPO_DIR}...")
    subprocess.run(["git", "clone", OFFICIAL_REPO_URL, str(OFFICIAL_REPO_DIR)], check=True)
else:
    print(f"Repository already exists at {OFFICIAL_REPO_DIR}, fetching updates...")
    subprocess.run(["git", "-C", str(OFFICIAL_REPO_DIR), "fetch", "--all"], check=True)

subprocess.run(["git", "-C", str(OFFICIAL_REPO_DIR), "checkout", PINNED_COMMIT_SHA], check=True)
actual_sha = subprocess.check_output(["git", "-C", str(OFFICIAL_REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
git_status = subprocess.check_output(["git", "-C", str(OFFICIAL_REPO_DIR), "status", "--porcelain"], text=True).strip()
assert actual_sha == PINNED_COMMIT_SHA, f"Expected commit {PINNED_COMMIT_SHA}, got {actual_sha}"

print(f"Verified Official F-OAL Commit SHA: {actual_sha}")
print(f"Working tree clean: {len(git_status) == 0}")
print("\nRecent commits in official repository:")
print(subprocess.check_output(["git", "-C", str(OFFICIAL_REPO_DIR), "log", "--oneline", "-5"], text=True))


In [ ]:
# === CELL 3: Install & Verify Dependencies ===
# Note: The official requirements.txt pins torch==1.13.1 and timm==0.9.7 from 2023.
# On modern Colab (Python 3.11+/3.12+ with CUDA 12/13), we preserve Colab's GPU-enabled
# torch/torchvision and install the required scientific & vision packages.
import subprocess
import sys

packages = [
    "timm>=0.9.7",
    "kornia>=0.7.0",
    "scikit-learn",
    "scipy",
    "PyYAML",
    "tqdm",
    "wandb",
    "matplotlib",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

import timm
import kornia
import scipy
import sklearn
import yaml
import torchvision
import wandb

print("Installed Library Versions:")
print(f"  torch        : {torch.__version__}")
print(f"  torchvision  : {torchvision.__version__}")
print(f"  timm         : {timm.__version__}")
print(f"  kornia       : {kornia.__version__}")
print(f"  scipy        : {scipy.__version__}")
print(f"  scikit-learn : {sklearn.__version__}")
print(f"  wandb        : {wandb.__version__}")


In [ ]:
# === CELL 4: Dataset Setup & Verification (CIFAR-100) ===
# In official-F-OAL/continuum/dataset_scripts/dataset_base.py:
# self.root = os.path.join("./datasets", self.dataset) -> /content/official-F-OAL/datasets/cifar100
import torchvision.datasets as tv_datasets
import numpy as np

cifar_root = OFFICIAL_REPO_DIR / "datasets" / "cifar100"
cifar_root.mkdir(parents=True, exist_ok=True)

train_set = tv_datasets.CIFAR100(root=str(cifar_root), train=True, download=True)
test_set = tv_datasets.CIFAR100(root=str(cifar_root), train=False, download=True)

assert len(train_set.data) == 50000, f"Expected 50,000 train images, got {len(train_set.data)}"
assert len(test_set.data) == 10000, f"Expected 10,000 test images, got {len(test_set.data)}"
assert len(set(train_set.targets)) == 100, "Expected 100 unique train classes"
assert len(set(test_set.targets)) == 100, "Expected 100 unique test classes"

print(f"CIFAR-100 ready at : {cifar_root}")
print(f"Train data shape   : {train_set.data.shape} (classes: {len(set(train_set.targets))})")
print(f"Test data shape    : {test_set.data.shape} (classes: {len(set(test_set.targets))})")


In [ ]:
# === CELL 5: Official Code Sanity Check (Zero Modification to Author Repo) ===
import sys
if str(OFFICIAL_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_REPO_DIR))

from models.pretrained import Encoder
from utils.setup_elements import transforms_match

with torch.no_grad():
    vit_ref = timm.create_model("vit_base_patch16_224", pretrained=True)
    pretrained_cfg = getattr(vit_ref, "pretrained_cfg", {})
    del vit_ref

    sanity_encoder = Encoder(n_classes=100, projection_size=1000).cuda()
    sanity_encoder.eval()
    proj_weight = sanity_encoder.linear_projection.weight.detach().cpu()
    dummy_x = torch.rand(2, 3, 224, 224, device="cuda")
    dummy_logits, dummy_act = sanity_encoder(dummy_x)

print("Official Encoder Sanity Check PASSED:")
print(f"  timm model name        : vit_base_patch16_224")
print(f"  timm hf_hub_id / tag   : {pretrained_cfg.get('hf_hub_id', 'N/A')} (tag: {pretrained_cfg.get('tag', 'N/A')})")
print(f"  linear_projection shape: {tuple(proj_weight.shape)} (min={proj_weight.min():.5f}, max={proj_weight.max():.5f}, std={proj_weight.std():.5f})")
print(f"  cifar100 transforms    : {transforms_match['cifar100']}")
print(f"  dummy_act shape        : {tuple(dummy_act.shape)} (min={dummy_act.min():.4f}, max={dummy_act.max():.4f})")
print(f"  dummy_logits shape     : {tuple(dummy_logits.shape)}")

del sanity_encoder, dummy_x, dummy_logits, dummy_act
torch.cuda.empty_cache()


In [ ]:
# === CELL 6: Full Official F-OAL Reproduction on CIFAR-100 ===
# Runs the unmodified official entrypoint general_main.py.
# Official defaults in general_main.py: --data cifar100 --cl_type nc --agent FOAL --num_runs 1 --seed 0
import time
import subprocess

OFFICIAL_SEED = 0          # Set to 0 (author default) or 42 (to match our seed 42)
OFFICIAL_NUM_RUNS = 1      # Set to 1 for single-seed or 3 for 3-run average
STDOUT_LOG_PATH = RESULTS_OUT_DIR / "official_stdout.log"

cmd = [
    sys.executable,
    "-u",
    "general_main.py",
    "--data", "cifar100",
    "--cl_type", "nc",
    "--agent", "FOAL",
    "--batch", "10",
    "--test_batch", "10",
    "--projection_size", "1000",
    "--num_runs", str(OFFICIAL_NUM_RUNS),
    "--seed", str(OFFICIAL_SEED),
    "--verbose", "True",
]

print(f"Executing command in {OFFICIAL_REPO_DIR}:")
print(" ".join(cmd))
print("=" * 80)

t_start = time.time()
with open(STDOUT_LOG_PATH, "w", encoding="utf-8") as log_f:
    proc = subprocess.Popen(
        cmd,
        cwd=str(OFFICIAL_REPO_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
        log_f.write(line)
    proc.wait()
total_duration_seconds = round(time.time() - t_start, 2)

if proc.returncode != 0:
    raise RuntimeError(f"Official F-OAL run failed with exit code {proc.returncode}")

print("=" * 80)
print(f"Official F-OAL completed successfully in {total_duration_seconds:.2f} seconds.")


In [ ]:
# === CELL 7: Standardized Results Extraction (metrics.json, metadata.json, task_accuracy.csv) ===
import csv
import json
import re
import numpy as np

log_text = STDOUT_LOG_PATH.read_text(encoding="utf-8")

# Parse per-stage accuracy rows printed by agents/base.py:
# "----------run 0 task 0 acc_array : [0.955 0. ... ]----------"
pattern = re.compile(r"run\s+(\d+)\s+task\s+(\d+)\s+acc_array\s*:\s*\[([^\]]+)\]")
runs_matrix = {}
for match in pattern.finditer(log_text):
    run_idx = int(match.group(1))
    task_idx = int(match.group(2))
    vals = [float(x) for x in match.group(3).strip().split()]
    runs_matrix.setdefault(run_idx, {})[task_idx] = vals

assert 0 in runs_matrix and len(runs_matrix[0]) == 10, (
    f"Expected 10 task accuracy rows in stdout log, found {len(runs_matrix.get(0, {}))}"
)

# Full 10x10 matrix for run 0 and lower-triangular seen-task matrix
full_10x10 = np.array([runs_matrix[0][t] for t in range(10)], dtype=np.float64)
task_matrix = [[round(float(full_10x10[i, j]), 6) for j in range(i + 1)] for i in range(10)]

stage_accuracies = [float(np.mean(row)) for row in task_matrix]
average_incremental_accuracy = float(np.mean(stage_accuracies))
last_accuracy = float(stage_accuracies[-1])

# Standard CL forgetting (over T-1 = 9 past tasks, matching src/foal/metrics.py)
task_forgetting_9 = []
for j in range(9):
    best_prev = max(task_matrix[i][j] for i in range(j, 9))
    task_forgetting_9.append(float(best_prev - task_matrix[9][j]))
forgetting_9_tasks = float(np.mean(task_forgetting_9))

# Official repo forgetting formula in experiment/metrics.py (averages over all T = 10 tasks including task 10 = 0.0)
best_all_10 = np.max(full_10x10, axis=0)
official_forgetting_10_tasks = float(np.mean(best_all_10 - full_10x10[-1, :]))

# Copy official cifar100_FOAL.csv into RESULTS_OUT_DIR
official_csv_src = OFFICIAL_REPO_DIR / "cifar100_FOAL.csv"
if official_csv_src.exists():
    shutil.copy2(official_csv_src, RESULTS_OUT_DIR / "cifar100_FOAL.csv")

metrics_payload = {
    "task_matrix": task_matrix,
    "average_incremental_accuracy": average_incremental_accuracy,
    "last_accuracy": last_accuracy,
    "forgetting": forgetting_9_tasks,
    "official_forgetting_10_tasks": official_forgetting_10_tasks,
    "num_tasks": 10,
    "stage_accuracies": stage_accuracies,
    "task_forgetting": task_forgetting_9,
    "final_task_accuracies": task_matrix[-1],
    "total_duration_seconds": total_duration_seconds,
}

metadata_payload = {
    "paper": "F-OAL NeurIPS 2024",
    "method": "official_foal",
    "experiment_type": "official_reproduction",
    "run_name": f"official_foal_cifar100_seed{OFFICIAL_SEED}",
    "official_repo_url": OFFICIAL_REPO_URL,
    "official_commit_sha": actual_sha,
    "seeds": {
        "base": OFFICIAL_SEED,
        "class_order": OFFICIAL_SEED,
        "projection": OFFICIAL_SEED,
        "stream": OFFICIAL_SEED,
    },
    "model_config": {
        "backbone": "vit_base_patch16_224",
        "pretrained": True,
        "feature_fusion": True,
        "smooth_projection": True,
        "projection_dim": 1000,
        "gamma": 1.0,
        "rls_dtype": "float32",
    },
    "training_protocol": {
        "dataset": "cifar100",
        "num_classes": 100,
        "classes_per_task": 10,
        "max_tasks": 10,
        "limit_train_per_task": None,
        "limit_test_per_task": None,
        "batch_size": 10,
        "device": "cuda",
        "one_pass": True,
    },
    "environment": {
        "python": sys.version,
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "timm": timm.__version__,
        "kornia": kornia.__version__,
        "gpu_name": torch.cuda.get_device_name(0),
        "cuda_version": torch.version.cuda,
    },
}

(RESULTS_OUT_DIR / "metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
(RESULTS_OUT_DIR / "metadata.json").write_text(json.dumps(metadata_payload, indent=2), encoding="utf-8")

with open(RESULTS_OUT_DIR / "task_accuracy.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["stage", "eval_task", "accuracy"])
    for i, row in enumerate(task_matrix, start=1):
        for j, val in enumerate(row, start=1):
            writer.writerow([i, j, f"{val:.6f}"])

print(f"Extracted Official F-OAL Metrics (Seed {OFFICIAL_SEED}):")
print(f"  Average Incremental Accuracy (A_bar) : {average_incremental_accuracy * 100:.2f}%")
print(f"  Last Accuracy (A_10)                 : {last_accuracy * 100:.2f}%")
print(f"  Forgetting (Standard 9 past tasks)   : {forgetting_9_tasks * 100:.2f}%")
print(f"  Forgetting (Author 10-task formula)  : {official_forgetting_10_tasks * 100:.2f}%")
print(f"  Total Runtime                        : {total_duration_seconds:.1f}s")


In [ ]:
# === CELL 8: Non-Intrusive W&B Experiment Logging ===
# Logs the extracted Official Code metrics and artifacts without modifying the author repository.
# Defaults to offline mode if WANDB_API_KEY is not set; syncable via `wandb sync`.
import os
import wandb

WANDB_PROJECT = "hcmus-foal-reproduction"
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", None)
WANDB_MODE = "online" if os.environ.get("WANDB_API_KEY") else "offline"

run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=f"official_foal_cifar100_seed{OFFICIAL_SEED}",
    job_type="official_reproduction",
    mode=WANDB_MODE,
    dir=str(RESULTS_OUT_DIR),
    config=metadata_payload,
)

for stage_idx, stage_acc in enumerate(stage_accuracies, start=1):
    wandb.log({"stage": stage_idx, "stage_avg_accuracy": stage_acc}, step=stage_idx)

wandb.summary["average_incremental_accuracy"] = average_incremental_accuracy
wandb.summary["last_accuracy"] = last_accuracy
wandb.summary["forgetting"] = forgetting_9_tasks
wandb.summary["official_forgetting_10_tasks"] = official_forgetting_10_tasks
wandb.summary["total_duration_seconds"] = total_duration_seconds

artifact = wandb.Artifact(
    name=f"official-foal-cifar100-seed{OFFICIAL_SEED}",
    type="evaluation_results",
)
for fname in ["metrics.json", "metadata.json", "task_accuracy.csv", "official_stdout.log", "cifar100_FOAL.csv"]:
    fpath = RESULTS_OUT_DIR / fname
    if fpath.exists():
        artifact.add_file(str(fpath), name=fname)
run.log_artifact(artifact)
run.finish()
print(f"W&B logging finished in mode={WANDB_MODE}. Run directory stored under {RESULTS_OUT_DIR / 'wandb'}")


In [ ]:
# === CELL 9: Package & Strictly Verify Archive (/content/foal_official_cifar100_results.zip) ===
import zipfile

ZIP_PATH = BASE_DIR / "foal_official_cifar100_results.zip"
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in sorted(RESULTS_OUT_DIR.rglob("*")):
        if file_path.is_file():
            arcname = pathlib.Path("official_foal_cifar100") / file_path.relative_to(RESULTS_OUT_DIR)
            zf.write(file_path, arcname=str(arcname))

# Mandatory Integrity Checks before Download
assert ZIP_PATH.exists(), "Archive was not created!"
zip_size_bytes = ZIP_PATH.stat().st_size
assert zip_size_bytes > 1024, f"Archive is unexpectedly small ({zip_size_bytes} bytes <= 1 KB)!"

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    bad_file = zf.testzip()
    assert bad_file is None, f"CRC check failed on {bad_file}"
    namelist = zf.namelist()
    required_entries = [
        "official_foal_cifar100/metrics.json",
        "official_foal_cifar100/metadata.json",
        "official_foal_cifar100/task_accuracy.csv",
        "official_foal_cifar100/official_stdout.log",
    ]
    for req in required_entries:
        assert req in namelist, f"Missing required entry in archive: {req}"
        assert zf.getinfo(req).file_size > 0, f"Entry {req} is 0 bytes!"

print(f"Archive verified successfully: {ZIP_PATH} ({zip_size_bytes:,} bytes, {len(namelist)} files, 0 CRC errors)")


In [ ]:
# === CELL 10: Three-Way Comparison (Paper vs Official Repo Run vs Our Implementation Run) ===
paper_metrics = {"avg_acc": 91.10, "last_acc": 86.50, "fgt": 5.50, "runtime": 261.0}
our_seed42_metrics = {"avg_acc": 79.33, "last_acc": 70.92, "fgt": 9.21, "runtime": 1735.3}
official_run_metrics = {
    "avg_acc": average_incremental_accuracy * 100.0,
    "last_acc": last_accuracy * 100.0,
    "fgt": forgetting_9_tasks * 100.0,
    "fgt_10": official_forgetting_10_tasks * 100.0,
    "runtime": total_duration_seconds,
}

print("### Three-Way Reproduction Comparison (CIFAR-100, 10 Tasks x 10 Classes)\n")
print("| Metric | Paper (3 runs) | Official Repo (Run) | Our Code (Seed 42) | Official - Paper (pp) | Ours - Official (pp) |")
print("|---|---:|---:|---:|---:|---:|")
for label, key in [
    ("Average Incremental Accuracy (%)", "avg_acc"),
    ("Last Accuracy (%)", "last_acc"),
    ("Forgetting Rate (9-task std, %)", "fgt"),
]:
    p_val = paper_metrics[key]
    o_val = official_run_metrics[key]
    u_val = our_seed42_metrics[key]
    print(f"| {label} | {p_val:.2f}% | {o_val:.2f}% | {u_val:.2f}% | {o_val - p_val:+.2f} pp | {u_val - o_val:+.2f} pp |")
print(
    f"| Official 10-task Forgetting (%) | 5.50% | {official_run_metrics['fgt_10']:.2f}% | 8.29% | "
    f"{official_run_metrics['fgt_10'] - 5.50:+.2f} pp | {8.29 - official_run_metrics['fgt_10']:+.2f} pp |"
)
print(
    f"| Runtime (seconds) | {paper_metrics['runtime']:.1f}s (RTX 4070 Ti) | "
    f"{official_run_metrics['runtime']:.1f}s ({torch.cuda.get_device_name(0)}) | "
    f"{our_seed42_metrics['runtime']:.1f}s (Tesla T4) | — | — |"
)
